In [4]:
%pip install composable

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import composable.records as rec

In [7]:
# Standard imports
import polars as pl
import polars.selectors as cs
import seaborn as sns
import numpy as np

from sklearn.preprocessing import StandardScaler

# Pipeline
from sklearn.pipeline import Pipeline


# Preprocessing stuff
from sklearn.preprocessing import LabelEncoder

# Model selection stuff
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV

# Classic classifiers
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier


# Metrics to use on the test set
# metric(y_test, y_predict)
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, roc_auc_score




# Trees and Forests in `sklearn`

In this notebook, we will cover the basics of classification, including:

1. The `DecisionTreeClassifier` and `RandomForestClassifier`.
2. Exploring tuning parameters for each model.
3. Performing a grid search.
4. Additional features of Random Forests.
5. Multiclass problems.

## Classification Example: Kyphosis Data Set

**Problem Statement:**
Given a DataSet of 81 patients who have undergone a spinal surgery for a deformation and the data if the condition recurred, Build a claddification Model to predict whether a patient being admitted for the surgery has chance for recurrence. This model will help the surgeons to plan appropriate level of treatment to prevent recurrence.

**Data Set Description :**
The kyphosis data frame has 81 rows and 4 columns. representing data on children who have had corrective spinal surgery

This data frame contains the following columns/Features:

1. *Kyphosis*: a factor with levels absent present indicating if a kyphosis (a type of deformation) was present after the operation.
2. *Age*: in months
3. *Number*: the number of vertebrae involved
4. *Start*: the number of the first (topmost) vertebra operated on.

**Research Question:** Can we predict whether if kyphosis will be present or absent after the surgery based on the age of the patient, number of vertebrae involved and the first vertebra operated on?

In [11]:
(kyphosis :=
 pl.read_csv('sample_data/kyphosis.csv')
   .drop('rownames')
)

Kyphosis,Age,Number,Start
str,i64,i64,i64
"""absent""",71,3,5
"""absent""",158,3,14
"""present""",128,4,5
"""absent""",2,5,1
"""absent""",1,4,15
…,…,…,…
"""present""",157,3,13
"""absent""",26,7,13
"""absent""",120,2,13


In [12]:
(X_kyphosis :=
 kyphosis
 .drop('Kyphosis')
 .to_pandas()
)

,Age,Number,Start
0,71,3,5
1,158,3,14
2,128,4,5
3,2,5,1
4,1,4,15
...,...,...,...
76,157,3,13
77,26,7,13
78,120,2,13
79,42,7,6


In [13]:
(y_kyphosis :=
 kyphosis
 .get_column('Kyphosis')
 .to_numpy()
 .ravel()
)

array(['absent', 'absent', 'present', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'present', 'present', 'absent',
       'absent', 'absent', 'absent', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'present', 'present', 'absent',
       'present', 'absent', 'absent', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'absent', 'absent', 'absent',
       'absent', 'present', 'absent', 'present', 'present', 'absent',
       'absent', 'absent', 'absent', 'present', 'absent', 'absent',
       'present', 'absent', 'absent', 'absent', 'present', 'absent',
       'absent', 'absent', 'absent', 'present', 'absent', 'absent',
       'present', 'present', 'absent', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'absent', 'present', 'absent',
       'absent', 'present', 'absent'], dtype=object)

## Topic 1 - Tree and Forest models in `sklearn`

`sklearn` comes with both a tree and forest based classifier, which can be found in the `tree` and `ensemble` submodules, respectively.

In [14]:
(tree := DecisionTreeClassifier(class_weight='balanced')
)

DecisionTreeClassifier(class_weight='balanced')

In [15]:
(forest := RandomForestClassifier(class_weight='balanced')
)

RandomForestClassifier(class_weight='balanced')

## Tuning parameters for trees and forests

First, let's inspect the tuning parameters for each model.

#### Exploring trees

In [ ]:
?tree

Type:        DecisionTreeClassifier
String form: DecisionTreeClassifier(class_weight='balanced')
File:        c:\users\lr7273ow\appdata\local\anaconda3\envs\polars\lib\site-packages\sklearn\tree\_classes.py
Docstring:  
A decision tree classifier.

Read more in the :ref:`User Guide <tree>`.

Parameters
----------
criterion : {"gini", "entropy", "log_loss"}, default="gini"
    The function to measure the quality of a split. Supported criteria are
    "gini" for the Gini impurity and "log_loss" and "entropy" both for the
    Shannon information gain, see :ref:`tree_mathematical_formulation`.

splitter : {"best", "random"}, default="best"
    The strategy used to choose the split at each node. Supported
    strategies are "best" to choose the best split and "random" to choose
    the best random split.

max_depth : int, default=None
    The maximum depth of the tree. If None, then nodes are expanded until
    all leaves are pure or until all leaves contain less than
    min_samples_split 

In [16]:
tree.get_params()

{'ccp_alpha': 0.0,
 'class_weight': 'balanced',
 'criterion': 'gini',
 'max_depth': None,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'random_state': None,
 'splitter': 'best'}

In [17]:
(tree_grid :=
 {'min_samples_leaf': [1,2,3],
  'min_samples_split': [2,3,4],
  'max_depth': [3,4,5],
 }
)

{'min_samples_leaf': [1, 2, 3],
 'min_samples_split': [2, 3, 4],
 'max_depth': [3, 4, 5]}

In [18]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [19]:
grid_search_tree = GridSearchCV(tree, tree_grid, cv = folds, scoring='roc_auc', n_jobs=-1, verbose=1)

grid_search_tree.fit(X_kyphosis, y_kyphosis)


Fitting 5 folds for each of 27 candidates, totalling 135 fits


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=DecisionTreeClassifier(class_weight='balanced'),
             n_jobs=-1,
             param_grid={'max_depth': [3, 4, 5], 'min_samples_leaf': [1, 2, 3],
                         'min_samples_split': [2, 3, 4]},
             scoring='roc_auc', verbose=1)

In [20]:
?cross_validate

In [21]:
my_scores = ['accuracy', 'balanced_accuracy', 'roc_auc']

(tree_scores :=
 cross_validate(grid_search_tree, X_kyphosis, y_kyphosis,
                cv = folds,
                scoring=my_scores,
                verbose=0,
                n_jobs=-1,
                )
 >> rec.subset([f'test_{m}' for m in my_scores])
 >> rec.map(np.mean)
)

{'test_accuracy': np.float64(0.7176470588235294),
 'test_balanced_accuracy': np.float64(0.6576923076923077),
 'test_roc_auc': np.float64(0.7057692307692307)}

#### Exploring forests

In [13]:
?forest

Object `forest` not found.


In [22]:
forest.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': 'balanced',
 'criterion': 'gini',
 'max_depth': None,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': None,
 'verbose': 0,
 'warm_start': False}

In [23]:
(forest_grid :=
{'max_depth': [2,3,4,5],
 'min_samples_leaf': [1,2,3],
 'min_samples_split': [2,3,4],
 'n_estimators': [10, 100, 500],
 }
)

{'max_depth': [2, 3, 4, 5],
 'min_samples_leaf': [1, 2, 3],
 'min_samples_split': [2, 3, 4],
 'n_estimators': [10, 100, 500]}

In [24]:
grid_search_forest = GridSearchCV(forest, forest_grid, cv = folds, scoring='roc_auc', n_jobs=-1, verbose=1)

grid_search_forest.fit(X_kyphosis, y_kyphosis)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=RandomForestClassifier(class_weight='balanced'),
             n_jobs=-1,
             param_grid={'max_depth': [2, 3, 4, 5],
                         'min_samples_leaf': [1, 2, 3],
                         'min_samples_split': [2, 3, 4],
                         'n_estimators': [10, 100, 500]},
             scoring='roc_auc', verbose=1)

In [25]:
(forest_scores :=
 cross_validate(grid_search_tree, X_kyphosis, y_kyphosis,
                cv = folds,
                scoring=my_scores,
                verbose=0,
                n_jobs=-1,
                )
 >> rec.subset([f'test_{m}' for m in my_scores])
 >> rec.map(np.mean)
)


{'test_accuracy': np.float64(0.7176470588235294),
 'test_balanced_accuracy': np.float64(0.6576923076923077),
 'test_roc_auc': np.float64(0.6916666666666667)}

## Topic 3 - Extra random forest features

As you learned from your out of class videos, the fact that random forests uses bagging allows for extra functionality, including

1. A measure of variable importance, and
2. An out of the box objective measurement of model fit.

#### Variable importance

In [26]:
grid_search_forest.best_estimator_.feature_importances_ # Ugh! so unreadable!

array([0.32353627, 0.21716972, 0.459294  ])

In [27]:
(feat_imp :=
 pl.DataFrame({'columns':grid_search_forest.feature_names_in_,
               'importance':grid_search_forest.best_estimator_.feature_importances_},
             )
   .sort('importance', descending=True)
)


columns,importance
str,f64
"""Start""",0.459294
"""Age""",0.323536
"""Number""",0.21717


#### Out of bag error estimate

**Notes.**

1. You need to set `oob_score=True` when creating the forest to get this measure.
2. Computing the `oob_score` has some overhead, so it is best leave this off when performing a grid search.  Instead, refit the winning model with this flag after.

In [28]:
grid_search_forest.best_params_

{'max_depth': 2,
 'min_samples_leaf': 3,
 'min_samples_split': 3,
 'n_estimators': 100}

In [29]:
(winning_forest_w_oob_error :=
 RandomForestClassifier(oob_score=True,

                       **grid_search_forest.best_params_,
                      )

)

RandomForestClassifier(max_depth=2, min_samples_leaf=3, min_samples_split=3,
                       oob_score=True)

In [30]:
winning_forest_w_oob_error.fit(X_kyphosis, y_kyphosis)

winning_forest_w_oob_error.oob_score_

0.7901234567901234

## Topic 4 - Multiclass classification

In [31]:
(olive_oil :=
 pl.read_csv("sample_data/OliveOils.csv")
).head(2)

Area.name,palmitic,palmitoleic,strearic,oleic,linoleic,eicosanoic,linolenic
str,i64,i64,i64,i64,i64,i64,i64
"""North-Apulia""",1075,75,226,7823,672,36,60
"""North-Apulia""",1088,73,224,7709,781,31,61


In [32]:
(X_oil :=
 olive_oil
 .drop('Area.name')
 .to_pandas()
).head(2)

,palmitic,palmitoleic,strearic,oleic,linoleic,eicosanoic,linolenic
0,1075,75,226,7823,672,36,60
1,1088,73,224,7709,781,31,61


In [33]:
(y_oil :=
 olive_oil
 .select('Area.name')
 .to_numpy()
 .ravel()
)[:3]

array(['North-Apulia', 'North-Apulia', 'North-Apulia'], dtype=object)

In [34]:
X_train_oil, X_test_oil, y_train_oil, y_test_oil = train_test_split(X_oil, y_oil, test_size=0.3, random_state=42, stratify=y_oil)

In [35]:
grid_search_tree_MC = GridSearchCV(tree, tree_grid, cv = folds, scoring='balanced_accuracy', n_jobs=-1, verbose=0)

grid_search_tree_MC.fit(X_train_oil, y_train_oil)
grid_search_tree_MC.score(X_test_oil, y_test_oil)

np.float64(0.7979253089310016)

In [36]:
grid_search_forest_MC = GridSearchCV(forest, forest_grid, cv = folds, scoring='balanced_accuracy', n_jobs=-1, verbose=0)

grid_search_forest_MC.fit(X_train_oil, y_train_oil)
grid_search_forest_MC.score(X_test_oil, y_test_oil)

np.float64(0.8902578691952505)

## <font color='red'> Exercise 1 </font>

Do some sleuthing to determine the impact of various tuning parameters for both trees and forests on the model flexibility and overfitting.

<font color='orange'>
A summary of your findings here.
</font>

In [ ]:
?tree

Type:        DecisionTreeClassifier
String form: DecisionTreeClassifier(class_weight='balanced')
File:        c:\users\lr7273ow\appdata\local\anaconda3\envs\polars\lib\site-packages\sklearn\tree\_classes.py
Docstring:  
A decision tree classifier.

Read more in the :ref:`User Guide <tree>`.

Parameters
----------
criterion : {"gini", "entropy", "log_loss"}, default="gini"
    The function to measure the quality of a split. Supported criteria are
    "gini" for the Gini impurity and "log_loss" and "entropy" both for the
    Shannon information gain, see :ref:`tree_mathematical_formulation`.

splitter : {"best", "random"}, default="best"
    The strategy used to choose the split at each node. Supported
    strategies are "best" to choose the best split and "random" to choose
    the best random split.

max_depth : int, default=None
    The maximum depth of the tree. If None, then nodes are expanded until
    all leaves are pure or until all leaves contain less than
    min_samples_split 

In [33]:
tree.get_params()

{'ccp_alpha': 0.0,
 'class_weight': 'balanced',
 'criterion': 'gini',
 'max_depth': None,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'random_state': None,
 'splitter': 'best'}

In [ ]:
?forest

Type:        RandomForestClassifier
String form: RandomForestClassifier(class_weight='balanced')
File:        c:\users\lr7273ow\appdata\local\anaconda3\envs\polars\lib\site-packages\sklearn\ensemble\_forest.py
Docstring:  
A random forest classifier.

A random forest is a meta estimator that fits a number of decision tree
classifiers on various sub-samples of the dataset and uses averaging to
improve the predictive accuracy and control over-fitting.
Trees in the forest use the best split strategy, i.e. equivalent to passing
`splitter="best"` to the underlying :class:`~sklearn.tree.DecisionTreeClassifier`.
The sub-sample size is controlled with the `max_samples` parameter if
`bootstrap=True` (default), otherwise the whole dataset is used to build
each tree.

For a comparison between tree-based ensemble models see the example
:ref:`sphx_glr_auto_examples_ensemble_plot_forest_hist_grad_boosting_comparison.py`.

This estimator has native support for missing values (NaNs). During training,


 #### The impact of certain parameters for the tree classification model such as criterion, splitter and max_depth params. The ccp_alpha parameter is able to chop off branches that don't add predictive power uses the measure to the quality of a split, the min_samples_leaf is used to choose the split at each node, and the max_depth (which is the most important) deals with overfitting.

#### The impact of certain parameters for the forest classification model such as n_estimator determines  the number of trees, the max features how many limits each tree can look for a split, and the bootstrap as the tree uses sub-samples which averages out the mean between trees.  

## <font color="red"> Exercise 2 </font>

1. Use a combined grid search to compare the performance of trees and forests on the Pima Indian diabetes data set.  
2. Validate the winning model by computing CV scores on the test set.
3. Use random forests to explore the feature importance.  If a forest wasn't the winning model, redo the grid search to find the best forest.
4. Comment on your findings.

In [39]:
(
    diabetes :=
 pl.read_csv('sample_data/diabetes_raw.csv')
)

Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diabetes
i64,i64,i64,i64,i64,f64,f64,i64,str
6,148,72,35,0,33.6,0.627,50,"""Yes"""
1,85,66,29,0,26.6,0.351,31,"""No"""
8,183,64,0,0,23.3,0.672,32,"""Yes"""
1,89,66,23,94,28.1,0.167,21,"""No"""
0,137,40,35,168,43.1,2.288,33,"""Yes"""
…,…,…,…,…,…,…,…,…
10,101,76,48,180,32.9,0.171,63,"""No"""
2,122,70,27,0,36.8,0.34,27,"""No"""
5,121,72,23,112,26.2,0.245,30,"""No"""


In [40]:
(X_diabetes :=
    diabetes
    .drop('Diabetes')
    .to_pandas()
 )

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33
...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63
764,2,122,70,27,0,36.8,0.340,27
765,5,121,72,23,112,26.2,0.245,30
766,1,126,60,0,0,30.1,0.349,47


In [41]:
( y_diabetes :=
       diabetes
       .select('Diabetes')
       .to_pandas()
)

,Diabetes
0,Yes
1,No
2,Yes
3,No
4,Yes
...,...
763,No
764,No
765,No
766,Yes


In [42]:
X_train_diabetes, X_test_diabetes, y_train_diabetes, y_test_diabetes = train_test_split(X_diabetes,y_diabetes, test_size=0.3, random_state=42)

In [43]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [44]:
(tree := DecisionTreeClassifier(class_weight = 'balanced')
)

DecisionTreeClassifier(class_weight='balanced')

In [45]:
grid_search_forest = GridSearchCV(forest, forest_grid, cv = folds, scoring='roc_auc', n_jobs=-1, verbose=1)

grid_search_forest.fit(X_diabetes, y_diabetes)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=RandomForestClassifier(class_weight='balanced'),
             n_jobs=-1,
             param_grid={'max_depth': [2, 3, 4, 5],
                         'min_samples_leaf': [1, 2, 3],
                         'min_samples_split': [2, 3, 4],
                         'n_estimators': [10, 100, 500]},
             scoring='roc_auc', verbose=1)

In [46]:
(tree_grid :=
  { 'classifier': [DecisionTreeClassifier()], # Kept in a list to avoid that previous error!
    'classifier__min_samples_leaf': [1, 2, 3],
    'classifier__min_samples_split': [2, 3, 4],
    'classifier__max_depth': [3, 4, 5]
 }
)

{'classifier': [DecisionTreeClassifier()],
 'classifier__min_samples_leaf': [1, 2, 3],
 'classifier__min_samples_split': [2, 3, 4],
 'classifier__max_depth': [3, 4, 5]}

In [47]:
(forest_grid :=
{
    'classifier': [RandomForestClassifier()], # Added the 's' in classifier
    'classifier__max_depth': [2, 3, 4, 5],
    'classifier__min_samples_leaf': [1, 2, 3],
    'classifier__min_samples_split': [2, 3, 4],
    'classifier__n_estimators': [10, 100, 500]
}
)

{'classifier': [RandomForestClassifier()],
 'classifier__max_depth': [2, 3, 4, 5],
 'classifier__min_samples_leaf': [1, 2, 3],
 'classifier__min_samples_split': [2, 3, 4],
 'classifier__n_estimators': [10, 100, 500]}

In [48]:
(generic_cls :=
 Pipeline(steps = [
     ('scaler', StandardScaler()),
     ('classifier', None)  # <-- classifier goes here
 ])

)

Pipeline(steps=[('scaler', StandardScaler()), ('classifier', None)])

In [49]:
(combined_grid :=

 [forest_grid, tree_grid]


)

[{'classifier': [RandomForestClassifier()],
  'classifier__max_depth': [2, 3, 4, 5],
  'classifier__min_samples_leaf': [1, 2, 3],
  'classifier__min_samples_split': [2, 3, 4],
  'classifier__n_estimators': [10, 100, 500]},
 {'classifier': [DecisionTreeClassifier()],
  'classifier__min_samples_leaf': [1, 2, 3],
  'classifier__min_samples_split': [2, 3, 4],
  'classifier__max_depth': [3, 4, 5]}]

In [50]:
(combined_grid_search :=
 GridSearchCV(generic_cls,combined_grid, cv = folds, scoring='balanced_accuracy', verbose=1 )

 )

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('classifier', None)]),
             param_grid=[{'classifier': [RandomForestClassifier()],
                          'classifier__max_depth': [2, 3, 4, 5],
                          'classifier__min_samples_leaf': [1, 2, 3],
                          'classifier__min_samples_split': [2, 3, 4],
                          'classifier__n_estimators': [10, 100, 500]},
                         {'classifier': [DecisionTreeClassifier()],
                          'classifier__max_depth': [3, 4, 5],
                          'classifier__min_samples_leaf': [1, 2, 3],
                          'classifier__min_samples_split': [2, 3, 4]}],
             scoring='balanced_accuracy', verbose=1)

In [51]:
combined_grid_search = GridSearchCV(generic_cls, combined_grid, cv = folds, scoring='roc_auc', n_jobs=-1, verbose=1)

combined_grid_search.fit(X_train_diabetes, y_train_diabetes)

Fitting 5 folds for each of 135 candidates, totalling 675 fits


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('classifier', None)]),
             n_jobs=-1,
             param_grid=[{'classifier': [RandomForestClassifier()],
                          'classifier__max_depth': [2, 3, 4, 5],
                          'classifier__min_samples_leaf': [1, 2, 3],
                          'classifier__min_samples_split': [2, 3, 4],
                          'classifier__n_estimators': [10, 100, 500]},
                         {'classifier': [DecisionTreeClassifier()],
                          'classifier__max_depth': [3, 4, 5],
                          'classifier__min_samples_leaf': [1, 2, 3],
                          'classifier__min_samples_split': [2, 3, 4]}],
             scoring='roc_auc', verbose=1)

In [53]:
grid_search_tree.best_params_

{'max_depth': 4, 'min_samples_leaf': 3, 'min_samples_split': 3}

In [54]:
(combined_grid :=

 [forest_grid, tree_grid]


)

[{'classifier': [RandomForestClassifier()],
  'classifier__max_depth': [2, 3, 4, 5],
  'classifier__min_samples_leaf': [1, 2, 3],
  'classifier__min_samples_split': [2, 3, 4],
  'classifier__n_estimators': [10, 100, 500]},
 {'classifier': [DecisionTreeClassifier()],
  'classifier__min_samples_leaf': [1, 2, 3],
  'classifier__min_samples_split': [2, 3, 4],
  'classifier__max_depth': [3, 4, 5]}]

In [55]:
my_scores = ['accuracy', 'balanced_accuracy', 'roc_auc']



(diabetes_scores :=
 cross_validate(combined_grid_search, X_test_diabetes, y_test_diabetes,
                cv = folds,
                scoring=my_scores,
                verbose=0,
                n_jobs=-1,
                )
 >> rec.subset([f'test_{m}' for m in my_scores])
 >> rec.map(np.mean)
)



{'test_accuracy': np.float64(0.7273820536540241),
 'test_balanced_accuracy': np.float64(0.6447043010752689),
 'test_roc_auc': np.float64(nan)}

<font color="orange">
Your summary here.
</font>

### The best classifier out of the two options (random forest and decision tree) was the random forest classifier which was able to predict the classes with max depth of 5, classifier min sample leaf of 2, classifier min sample split of 2.  The cross validation map for our scores went as follows:
### test_accuracy: 0.71
### test_balanced_accuracy: 0.64
### test_roc_auc : NaN

## <font color="red"> Exercise 3 </font>

Now we build on the final exercise of the previous activity, where we performed a combined grid search over all of the classic classification methods.  Redo this, but this time add trees and forests of each type (alone, OvR, and OVO).

1. Create a combined grid on all 15 classifiers (5 classic classifiers + 5*OvR + 5*OvO).  
2. For `kNN`, add a number of `metric`s and `weights` to the grid.
3. Add 6 tree based classifiers to the grid search (tree + forest as well as OvR and OvO for each).
4. Perform the grid search over all the models and determing the winning model.
5. Evaluate the performance of the winning model using CV and write a summary interpreting each metric.

In [69]:
(knn_alone :=
  {'classifier': [KNeighborsClassifier()],
   'classifier__n_neighbors': list(range(2,12,2)),
    'classifier__metric': ['l1','l2'],
    'classifier__weights': ['uniform','distance']
  }
 )

{'classifier': [KNeighborsClassifier()],
 'classifier__n_neighbors': [2, 4, 6, 8, 10],
 'classifier__metric': ['l1', 'l2'],
 'classifier__weights': ['uniform', 'distance']}

In [70]:
knn_wrapped = {
    'classifier': [
        OneVsRestClassifier(KNeighborsClassifier()),
        OneVsOneClassifier(KNeighborsClassifier())
    ],
    'classifier__estimator__n_neighbors': list(range(2,12,2)),
    'classifier__estimator__metric': ['l1','l2'],
    'classifier__estimator__weights': ['uniform','distance']
}

In [ ]:
#(knn_wrapped :=
#    {'classifier': [OneVsOneClassifier(KNeighborsClassifier()),OneVsOneClassifier(KNeighborsClassifier())],
#     'classifier__estimator__n_neighbors': list(range(2,12,2)),
#     'classifier__estimator__metric': ['l1','l2'],
#     'classifier__estimator__weights': ['uniform','distance']
#    }
#)

{'classifier': [OneVsOneClassifier(estimator=KNeighborsClassifier()),
  OneVsOneClassifier(estimator=KNeighborsClassifier())],
 'classifier__estimator__n_neighbors': [2, 4, 6, 8, 10],
 'classifier__estimator__metric': ['l1', 'l2'],
 'classifier__estimator__weights': ['uniform', 'distance']}

In [71]:
(knn_grid :=
[knn_alone, knn_wrapped]
)

[{'classifier': [KNeighborsClassifier()],
  'classifier__n_neighbors': [2, 4, 6, 8, 10],
  'classifier__metric': ['l1', 'l2'],
  'classifier__weights': ['uniform', 'distance']},
 {'classifier': [OneVsRestClassifier(estimator=KNeighborsClassifier()),
   OneVsOneClassifier(estimator=KNeighborsClassifier())],
  'classifier__estimator__n_neighbors': [2, 4, 6, 8, 10],
  'classifier__estimator__metric': ['l1', 'l2'],
  'classifier__estimator__weights': ['uniform', 'distance']}]

In [72]:
# Your code here (add cells as needed.)


(nontuning_grid :=  {'classifier' :
[LogisticRegression(max_iter=10000),
 OneVsRestClassifier(LogisticRegression(max_iter=10000)),
 OneVsOneClassifier(LogisticRegression(max_iter=10000)),
 GaussianNB(),
 OneVsRestClassifier(GaussianNB()),
 OneVsOneClassifier(GaussianNB()),
 LinearDiscriminantAnalysis(),
 OneVsRestClassifier(LinearDiscriminantAnalysis()),
 OneVsOneClassifier(LinearDiscriminantAnalysis()),
 QuadraticDiscriminantAnalysis(),
 OneVsRestClassifier(QuadraticDiscriminantAnalysis()),
 OneVsOneClassifier(QuadraticDiscriminantAnalysis())

]
}
)

{'classifier': [LogisticRegression(max_iter=10000),
  OneVsRestClassifier(estimator=LogisticRegression(max_iter=10000)),
  OneVsOneClassifier(estimator=LogisticRegression(max_iter=10000)),
  GaussianNB(),
  OneVsRestClassifier(estimator=GaussianNB()),
  OneVsOneClassifier(estimator=GaussianNB()),
  LinearDiscriminantAnalysis(),
  OneVsRestClassifier(estimator=LinearDiscriminantAnalysis()),
  OneVsOneClassifier(estimator=LinearDiscriminantAnalysis()),
  QuadraticDiscriminantAnalysis(),
  OneVsRestClassifier(estimator=QuadraticDiscriminantAnalysis()),
  OneVsOneClassifier(estimator=QuadraticDiscriminantAnalysis())]}

In [ ]:
DecisionTreeClassifier().get_params()

{'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'random_state': None,
 'splitter': 'best'}

In [57]:
(
 tree_alone :=
 { 'classifier' : [DecisionTreeClassifier()],
    'classifier__max_depth': [3, 5, 10, 15, 20],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__min_samples_split': [2, 5, 10]
 }
)

{'classifier': [DecisionTreeClassifier()],
 'classifier__max_depth': [3, 5, 10, 15, 20],
 'classifier__min_samples_leaf': [1, 2, 4],
 'classifier__min_samples_split': [2, 5, 10]}

In [58]:
(
 tree_wrapped :=

 { 'classifier' : [OneVsRestClassifier(DecisionTreeClassifier()),
                   OneVsOneClassifier(DecisionTreeClassifier())],
    'classifier__estimator__max_depth': [3, 5, 10, 15, 20],
    'classifier__estimator__min_samples_leaf': [1, 2, 4],
    'classifier__estimator__min_samples_split': [2, 5, 10]
 }

)

{'classifier': [OneVsRestClassifier(estimator=DecisionTreeClassifier()),
  OneVsOneClassifier(estimator=DecisionTreeClassifier())],
 'classifier__estimator__max_depth': [3, 5, 10, 15, 20],
 'classifier__estimator__min_samples_leaf': [1, 2, 4],
 'classifier__estimator__min_samples_split': [2, 5, 10]}

In [59]:
(tree_grid :=

 [tree_alone,tree_wrapped]


)

[{'classifier': [DecisionTreeClassifier()],
  'classifier__max_depth': [3, 5, 10, 15, 20],
  'classifier__min_samples_leaf': [1, 2, 4],
  'classifier__min_samples_split': [2, 5, 10]},
 {'classifier': [OneVsRestClassifier(estimator=DecisionTreeClassifier()),
   OneVsOneClassifier(estimator=DecisionTreeClassifier())],
  'classifier__estimator__max_depth': [3, 5, 10, 15, 20],
  'classifier__estimator__min_samples_leaf': [1, 2, 4],
  'classifier__estimator__min_samples_split': [2, 5, 10]}]

In [60]:
(
 forest_alone := {
    'classifier': [RandomForestClassifier()],
    'classifier__n_estimators': [50, 100, 200, 500],
    'classifier__max_depth': [5, 10, 15, 20],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__min_samples_split': [2, 5, 10]
}

)

{'classifier': [RandomForestClassifier()],
 'classifier__n_estimators': [50, 100, 200, 500],
 'classifier__max_depth': [5, 10, 15, 20],
 'classifier__min_samples_leaf': [1, 2, 4],
 'classifier__min_samples_split': [2, 5, 10]}

In [62]:
(
 forest_wrapped := {
    'classifier': [OneVsRestClassifier(RandomForestClassifier()),
                    OneVsOneClassifier(RandomForestClassifier())],
    'classifier__estimator__n_estimators': [50, 100, 200, 500],
    'classifier__estimator__max_depth': [5, 10, 15, 20],
    'classifier__estimator__min_samples_leaf': [1, 2, 4],
    'classifier__estimator__min_samples_split': [2, 5, 10]
}

)

{'classifier': [OneVsRestClassifier(estimator=RandomForestClassifier()),
  OneVsOneClassifier(estimator=RandomForestClassifier())],
 'classifier__estimator__n_estimators': [50, 100, 200, 500],
 'classifier__estimator__max_depth': [5, 10, 15, 20],
 'classifier__estimator__min_samples_leaf': [1, 2, 4],
 'classifier__estimator__min_samples_split': [2, 5, 10]}

In [65]:
(forest_grid :=

    [forest_alone,forest_wrapped]

 )

[{'classifier': [RandomForestClassifier()],
  'classifier__n_estimators': [50, 100, 200, 500],
  'classifier__max_depth': [5, 10, 15, 20],
  'classifier__min_samples_leaf': [1, 2, 4],
  'classifier__min_samples_split': [2, 5, 10]},
 {'classifier': [OneVsRestClassifier(estimator=RandomForestClassifier()),
   OneVsOneClassifier(estimator=RandomForestClassifier())],
  'classifier__estimator__n_estimators': [50, 100, 200, 500],
  'classifier__estimator__max_depth': [5, 10, 15, 20],
  'classifier__estimator__min_samples_leaf': [1, 2, 4],
  'classifier__estimator__min_samples_split': [2, 5, 10]}]

In [ ]:
#(combined_grid :=

#[forest_grid,tree_grid,knn_grid,nontuning_grid]
#)

[[{'classifier': [RandomForestClassifier()],
   'classifier__n_estimators': [50, 100],
   'classifier__max_depth': [5, 10]},
  {'classifier': [OneVsRestClassifier(estimator=RandomForestClassifier()),
    OneVsOneClassifier(estimator=RandomForestClassifier())],
   'classifier__estimator__n_estimators': [50, 100]}],
 [{'classifier': [DecisionTreeClassifier()],
   'classifier__max_depth': [3, 5, 10]},
  {'classifier': [OneVsRestClassifier(estimator=DecisionTreeClassifier()),
    OneVsOneClassifier(estimator=DecisionTreeClassifier())],
   'classifier__estimator__max_depth': [3, 5, 10]}],
 [{'classifier': [KNeighborsClassifier()],
   'classifier__n_neighbors': [2, 4, 6, 8, 10],
   'classifier__metric': ['l1', 'l2'],
   'classifier__weights': ['uniform', 'distance']},
  {'classifier': [OneVsRestClassifier(estimator=KNeighborsClassifier()),
    OneVsOneClassifier(estimator=KNeighborsClassifier())],
   'classifier__estimator__n_neighbors': [2, 4, 6, 8, 10],
   'classifier__estimator__metric': 

In [73]:
(combined_grid :=

forest_grid + tree_grid + knn_grid + [nontuning_grid]
#[forest_grid,tree_grid,knn_grid,nontuning_grid]



)

[{'classifier': [RandomForestClassifier()],
  'classifier__n_estimators': [50, 100, 200, 500],
  'classifier__max_depth': [5, 10, 15, 20],
  'classifier__min_samples_leaf': [1, 2, 4],
  'classifier__min_samples_split': [2, 5, 10]},
 {'classifier': [OneVsRestClassifier(estimator=RandomForestClassifier()),
   OneVsOneClassifier(estimator=RandomForestClassifier())],
  'classifier__estimator__n_estimators': [50, 100, 200, 500],
  'classifier__estimator__max_depth': [5, 10, 15, 20],
  'classifier__estimator__min_samples_leaf': [1, 2, 4],
  'classifier__estimator__min_samples_split': [2, 5, 10]},
 {'classifier': [DecisionTreeClassifier()],
  'classifier__max_depth': [3, 5, 10, 15, 20],
  'classifier__min_samples_leaf': [1, 2, 4],
  'classifier__min_samples_split': [2, 5, 10]},
 {'classifier': [OneVsRestClassifier(estimator=DecisionTreeClassifier()),
   OneVsOneClassifier(estimator=DecisionTreeClassifier())],
  'classifier__estimator__max_depth': [3, 5, 10, 15, 20],
  'classifier__estimator__

In [74]:
(combined_grid_search :=
GridSearchCV(generic_cls, combined_grid,cv=folds, scoring='balanced_accuracy', verbose=1)
)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('classifier', None)]),
             param_grid=[{'classifier': [RandomForestClassifier()],
                          'classifier__max_depth': [5, 10, 15, 20],
                          'classifier__min_samples_leaf': [1, 2, 4],
                          'classifier__min_samples_split': [2, 5, 10],
                          'classifier__n_estimators': [50,...
                                         OneVsOneClassifier(estimator=GaussianNB()),
                                         LinearDiscriminantAnalysis(),
                                         OneVsRestClassifier(estimator=LinearDiscriminantAnalysis()),
                                         OneVsOneClassifier(estimator=LinearDiscriminantAnalysis()),
                                         QuadraticDiscriminantAnalysis(),
                                         OneVsRestClassifier(estimator=QuadraticDiscriminantAnalysis()),
                                         OneVsOneClassifier(estimator=QuadraticDiscriminantAnalysis())]}],
             scoring='balanced_accuracy', verbose=1)

In [75]:
(combined_grid_search.fit(X_train_diabetes, y_train_diabetes))

Fitting 5 folds for each of 639 candidates, totalling 3195 fits


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example usi

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('classifier', None)]),
             param_grid=[{'classifier': [RandomForestClassifier()],
                          'classifier__max_depth': [5, 10, 15, 20],
                          'classifier__min_samples_leaf': [1, 2, 4],
                          'classifier__min_samples_split': [2, 5, 10],
                          'classifier__n_estimators': [50,...
                                         OneVsOneClassifier(estimator=GaussianNB()),
                                         LinearDiscriminantAnalysis(),
                                         OneVsRestClassifier(estimator=LinearDiscriminantAnalysis()),
                                         OneVsOneClassifier(estimator=LinearDiscriminantAnalysis()),
                                         QuadraticDiscriminantAnalysis(),
                                         OneVsRestClassifier(estimator=QuadraticDiscriminantAnalysis()),
                                         OneVsOneClassifier(estimator=QuadraticDiscriminantAnalysis())]}],
             scoring='balanced_accuracy', verbose=1)

In [76]:
print(f"Total models searched: {len(combined_grid_search.cv_results_['params'])}")
import pandas as pd
df = pd.DataFrame(combined_grid_search.cv_results_)
# This sorts them so you can see if QDA is at #2 or #20
print(df[['param_classifier', 'mean_test_score']].sort_values('mean_test_score', ascending=False))

Total models searched: 639
                                      param_classifier  mean_test_score
328  OneVsOneClassifier(estimator=RandomForestClass...         0.744806
330  OneVsOneClassifier(estimator=RandomForestClass...         0.742142
414  OneVsOneClassifier(estimator=RandomForestClass...         0.740972
240  OneVsRestClassifier(estimator=RandomForestClas...         0.740900
76                            RandomForestClassifier()         0.740713
..                                                 ...              ...
617  OneVsOneClassifier(estimator=KNeighborsClassif...         0.641027
597  OneVsRestClassifier(estimator=KNeighborsClassi...         0.641027
587  OneVsRestClassifier(estimator=KNeighborsClassi...         0.639031
607  OneVsOneClassifier(estimator=KNeighborsClassif...         0.639031
567                             KNeighborsClassifier()         0.639031

[639 rows x 2 columns]


In [77]:
metrics = ['accuracy',
           'balanced_accuracy',
           'f1_micro'
]


(all_combined_test_scores :=
 cross_validate(combined_grid_search, X_test_diabetes,y_test_diabetes,
                cv = folds,
                scoring = metrics,
                verbose =1,
                n_jobs = -1
 )
  >>rec.subset([f'test_{m}' for m in metrics])
  >>rec.map(np.mean)
 )


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed: 66.6min finished


{'test_accuracy': np.float64(0.7448658649398705),
 'test_balanced_accuracy': np.float64(0.710752688172043),
 'test_f1_micro': np.float64(0.7448658649398705)}

### Overall, the best classifier of the 89 models was the Random Forest Classifier with a test score of 0.74, the parameters of the classifier including a n_estimator of 50, min samples split of 5 and a max depth of 10. The metrics for the cross validation of the combined grid search went as follows:


### 'test_accuracy':  0.75   
### 'test_balanced_accuracy' : 0.71
###  'test_f1_micro' : 0.75